# Análise Exploratória de Dados - `happyT` processado

Este notebook faz uma verificação simples das bases processadas de treino, validação e teste geradas pelo pré-processamento. O objetivo é conferir tamanhos, primeiras linhas, estatísticas rápidas e `describe` das features e do alvo transformado.

## 1. Carregamento dos dados processados

Nesta etapa, carregamos os arquivos `X_train`, `X_val`, `X_test`, `y_train`, `y_val` e `y_test` salvos em `data/processed/happyT`. As features já estão escaladas e o alvo `redshift` está em escala `log1p`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

In [ ]:
DATA_DIR = Path("..") / "data" / "processed" / "happyT"

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns")
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze("columns")
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze("columns")

datasets = {
    "treino": (X_train, y_train),
    "validacao": (X_val, y_val),
    "teste": (X_test, y_test),
}

print("Bases carregadas com sucesso.")

## 2. Diferenças de tamanho entre treino, validação e teste

Aqui comparamos a quantidade de linhas e colunas em cada partição. O conjunto de teste é maior porque combina os splits externos B, C e D.

In [ ]:
size_summary = []

for name, (X, y) in datasets.items():
    size_summary.append(
        {
            "base": name,
            "linhas_X": X.shape[0],
            "colunas_X": X.shape[1],
            "linhas_y": y.shape[0],
            "percentual_do_total": X.shape[0] / sum(data[0].shape[0] for data in datasets.values()),
        }
    )

size_summary = pd.DataFrame(size_summary)
size_summary["percentual_do_total"] = size_summary["percentual_do_total"] * 100

display(size_summary)

In [ ]:
for name, (X, y) in datasets.items():
    display(Markdown(f"### Shape da base de {name}"))
    print(f"X_{name}: {X.shape[0]:,} linhas x {X.shape[1]:,} colunas")
    print(f"y_{name}: {y.shape[0]:,} linhas")

## 3. Primeiras linhas das bases processadas

Abaixo exibimos `head(10)` para cada partição. As colunas devem permanecer com os mesmos nomes das features originais, mas seus valores já estão transformados pelo pré-processamento.

In [ ]:
for name, (X, y) in datasets.items():
    display(Markdown(f"### X_{name} - head(10)"))
    display(X.head(10))

    display(Markdown(f"### y_{name} - head(10)"))
    display(y.head(10).to_frame(name="redshift_log1p"))

## 4. Estatísticas rápidas

Nesta seção, verificamos valores nulos, infinitos, médias e desvios das features processadas. Como as magnitudes passaram por `StandardScaler`, esperamos médias próximas de zero no treino. Os erros passaram por `log1p` e `RobustScaler`, então são avaliados em uma escala robusta.

In [ ]:
quick_stats = []

for name, (X, y) in datasets.items():
    numeric_values = X.to_numpy()
    quick_stats.append(
        {
            "base": name,
            "nulos_X": int(X.isna().sum().sum()),
            "nulos_y": int(y.isna().sum()),
            "infinitos_X": int(np.isinf(numeric_values).sum()),
            "media_geral_X": X.mean().mean(),
            "desvio_medio_X": X.std().mean(),
            "min_y_log1p": y.min(),
            "media_y_log1p": y.mean(),
            "max_y_log1p": y.max(),
        }
    )

quick_stats = pd.DataFrame(quick_stats)
display(quick_stats)

In [ ]:
feature_mean_std = []

for name, (X, _) in datasets.items():
    stats = X.agg(["mean", "std"]).T.reset_index()
    stats.insert(0, "base", name)
    stats = stats.rename(columns={"index": "feature"})
    feature_mean_std.append(stats)

feature_mean_std = pd.concat(feature_mean_std, ignore_index=True)
display(feature_mean_std)

## 5. Describe das features processadas

O `describe` mostra a distribuição das features depois das transformações. Essa etapa ajuda a confirmar se treino, validação e teste estão em escalas compatíveis.

In [ ]:
for name, (X, _) in datasets.items():
    display(Markdown(f"### Describe de X_{name}"))
    display(X.describe().T)

## 6. Describe do alvo transformado

O alvo salvo está em escala `log1p(redshift)`. Para cálculo de métricas finais, as predições devem ser convertidas de volta para a escala original com `np.expm1` ou com a função `inverse_transform_target` do módulo de pré-processamento.

In [ ]:
for name, (_, y) in datasets.items():
    display(Markdown(f"### Describe de y_{name}"))
    display(y.to_frame(name="redshift_log1p").describe().T)

## 7. Conferência das colunas

Por fim, confirmamos que as partições possuem o mesmo conjunto de features e que variáveis como `id`, `redshift` e `redshiftErr` não aparecem em `X`.

In [ ]:
columns_summary = pd.DataFrame(
    {
        "X_train": X_train.columns,
        "X_val": X_val.columns,
        "X_test": X_test.columns,
    }
)

same_columns = list(X_train.columns) == list(X_val.columns) == list(X_test.columns)
forbidden_columns = {"id", "redshift", "redshiftErr"}
forbidden_present = forbidden_columns.intersection(X_train.columns)

print(f"Mesmas colunas em treino, validacao e teste: {same_columns}")
print(f"Colunas proibidas presentes em X_train: {sorted(forbidden_present)}")
display(columns_summary)